# Seasonality and Events on the M5 National Series

This notebook fits the national `TOTAL` series of `load_m5_aggregate()` with the #118 default M5 model: a level state
and six weekday states that drift, a Yeo-Johnson target and half-normal priors on its sigmas. It then adds an annual
Fourier pair and one column per M5 event, as covariates with fixed coefficients, and compares three fits:

| Fit | Added columns | Prior of each added coefficient |
|---|---|---|
| **no covariates** | none | none |
| **loose** | annual Fourier and events | $N(0, 1)$ |
| **tight** | the same columns | $N(0, 0.1^2)$ |

The last section splits each fit's forecast into its level, day of week, annual Fourier and events. This is not a
scored stage: it shows how the covariates behave on one dense series and one 28-day window. The notebook is committed
without outputs. It reads only the packaged panel and needs the `m5` extra: `pip install -e ".[m5]"`.

## 1. Libraries and data

`load_m5_aggregate()` holds 756 days of ten series. Its last 28 days, d_1914 to d_1941, are M5's validation window,
which #143 reads only once per candidate. The notebook drops those days as soon as it loads the panel and holds out the
pre-validation window of #125 instead, d_1886 to d_1913. That leaves 700 days to fit on and 28 to score.

In [ ]:
import jax.numpy as jnp
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from bunobee.datasets import load_m5_aggregate
from bunobee.models.m5.m5_ssp_optim import SigmaPrior, fit_one_series_opt, predict_one_series_opt
from bunobee.models.m5.transforms import PowerTransform
from bunobee.regression import make_periodic_dummies

plt.rcParams.update(
    {
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#e6e5e0",
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "font.size": 10,
        "axes.titlesize": 11,
        "lines.linewidth": 1.5,
    }
)

# One color per fit, the same in every figure. Actual sales are black.
FIT_COLORS = {"no covariates": "#2a78d6", "loose": "#eb6834", "tight": "#1baf7a"}
ACTUAL_COLOR = "#0b0b0b"
CONTEXT_COLOR = "#b5b4ad"

In [ ]:
N_FIT = 700
HORIZON = 28
M5_FIRST_DATE = pd.Timestamp("2011-01-29")  # d_1

panel = load_m5_aggregate()
last_day_read = np.sort(panel["date"].unique())[N_FIT + HORIZON - 1]
panel = panel[panel["date"] <= last_day_read]  # No later cell can read the validation window.

days = panel[panel["series_id"] == "TOTAL"].set_index("date")  # One row per date, with the calendar columns.
sales = days["sales"].to_numpy(dtype=float)
dates = days.index

d_number = (dates - M5_FIRST_DATE).days + 1
assert len(dates) == N_FIT + HORIZON and (d_number[N_FIT], d_number[-1]) == (1886, 1913)
print(f"fit:     {dates[0].date()} to {dates[N_FIT - 1].date()}, d_{d_number[0]} to d_{d_number[N_FIT - 1]}")
print(f"holdout: {dates[N_FIT].date()} to {dates[-1].date()}, d_{d_number[N_FIT]} to d_{d_number[-1]}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(dates, sales, color=ACTUAL_COLOR, lw=0.8)
ax.axvspan(dates[N_FIT], dates[-1], color="#f0efe9", zorder=0)
ax.annotate("holdout", (dates[N_FIT], 1), xycoords=("data", "axes fraction"), va="top", color="#52514e")
ax.set_title("TOTAL daily unit sales")
ax.set_ylabel("units")
fig.tight_layout()
plt.show()

## 2. The design

The base columns are the default's: a column of ones for the level, then Tuesday to Sunday with Monday dropped. Those
are in every fit. The covariates are the two groups below, built from the panel's own columns:

- **Annual:** the sine and cosine of period 365.25 days, the first annual Fourier pair.
- **Events:** one 0/1 column per name in `event_name_1` and `event_name_2`.

In [ ]:
base = make_periodic_dummies(len(dates), period=7, phase=int(dates[0].dayofweek), intercept=True)
n_base = base.shape[1]

event_names = np.setdiff1d(days[["event_name_1", "event_name_2"]], [""]).tolist()
events = pd.DataFrame({n: (days["event_name_1"] == n) | (days["event_name_2"] == n) for n in event_names}).astype(float)
calendar = pd.concat({"annual": days[["annual_sin_1", "annual_cos_1"]], "event": events}, axis=1)

COV_GROUPS = np.array(calendar.columns.get_level_values(0))
COV_NAMES = np.array(calendar.columns.get_level_values(1))
covariates = calendar.to_numpy()  # (days, n_covariates)
N_COVARIATES = covariates.shape[1]

print(f"base columns: {n_base} (level, then Tuesday to Sunday)")
print(f"covariates:   {N_COVARIATES}")
pd.Series(COV_GROUPS).value_counts(sort=False).rename("columns").to_frame()

The events, each with its number of fit days and holdout days. The holdout has **no event day**, so the event columns
reach its forecasts only through what they take away from the other states, and it can't say whether their
coefficients forecast well. Some events are seen once in the fit days.

In [ ]:
event_days = pd.DataFrame(
    {"fit days": events.iloc[:N_FIT].sum().astype(int), "holdout days": events.iloc[N_FIT:].sum().astype(int)}
)
print(f"event days in the holdout: {event_days['holdout days'].sum()}")
event_days.T

## 3. Fits with no covariates, a loose prior and a tight prior

The model, on the standardized Yeo-Johnson scale of the series, is

$$y_t = \ell_t + w_t^\top d_t + \beta^\top x_t + \varepsilon_t, \qquad \beta_j \sim N(0, \tau^2),$$

with the level $\ell_t$, the weekday states $w_t$ on the weekday columns $d_t$, and the fixed coefficients $\beta$ on
the covariates $x_t$. $\tau$ is the only thing that changes between the loose and the tight fit:

- **Loose, $\tau = 1$:** the target has a standard deviation of 1 over the fit days, so a single coefficient may be as
  large as the whole series' variation.
- **Tight, $\tau = 0.1$:** one tenth of that. About 95% of the prior's mass lies within $\pm 0.2$ of the target's
  standard deviation, well inside the daily noise. A coefficient needs many days behind it to move far from zero.

`fit_one_series_opt` has no argument for $\tau$, since it gives every state `a0 = 0` and `P0 = 1`, but it needs none:
a column $\tau x$ with a coefficient $b \sim N(0, 1)$ is the same model as a column $x$ with a coefficient
$\beta = \tau b \sim N(0, \tau^2)$. So the notebook scales the covariate columns by $\tau$ and multiplies the fitted
coefficients back by $\tau$.

Each fit is a MAP fit on the 700 fit days: Adam chooses the sigmas against the filter's likelihood and the filter
updates the coefficients day by day. The forecast uses the states filtered up to the last fit day.

In [ ]:
TRANSFORM = PowerTransform("yeo-johnson")
# The half-normal priors of the #118 default (HALF_NORMAL_PRIORS in playground/m5_prototype/m5_models.py).
SIGMA_PRIORS = {"sigma_h_prior": SigmaPrior("half_normal", (1.0,)), "sigma_q_prior": SigmaPrior("half_normal", (0.05,))}

FITS = {"no covariates": None, "loose": 1.0, "tight": 0.1}


def design(tau=None):
    # The design over every day, with its covariates scaled by tau. tau=None leaves them out.
    return base if tau is None else np.column_stack([base, tau * covariates])


def fit_total(tau=None):
    # One MAP fit of TOTAL. `states` are the filtered states of every fit day, with the coefficients on the unscaled
    # columns (beta = tau * b).
    Z = design(tau).astype(np.float32)
    fit = fit_one_series_opt(
        sales[:N_FIT],
        Z=jnp.asarray(Z[:N_FIT]),
        transform=TRANSFORM,
        fixed_states=None if tau is None else np.arange(Z.shape[1]) >= n_base,
        **SIGMA_PRIORS,
    )
    states = fit["at"].copy()
    states[:, n_base:] *= tau or 1.0
    return {
        "forecast": predict_one_series_opt(fit, Z_future=Z[N_FIT:]),
        "states": states,
        "sigma_h": fit["sigma_h"],
        "sigma_q_level": fit["sigma_q"][0],
        "loss": fit["final_loss"],
    }


results = {name: fit_total(tau) for name, tau in FITS.items()}

MAPE and bias on the 28 holdout days. Bias is the error of the 28-day sum. The seasonal-naive row repeats the last fit
week, the baseline of `tests/test_m5_accuracy.py`. `loss` is the fit's negative log posterior of the sigmas, which sees
only the fit days.

In [ ]:
actual = sales[N_FIT:]
naive = sales[N_FIT - 7 : N_FIT][np.arange(HORIZON) % 7]


def mape(forecast):
    return 100 * np.mean(np.abs(forecast - actual) / actual)


def bias(forecast):
    return 100 * (forecast.sum() / actual.sum() - 1)


accuracy = pd.DataFrame(
    {
        name: {
            "MAPE %": mape(result["forecast"]),
            "bias %": bias(result["forecast"]),
            "sigma_h": result["sigma_h"],
            "level drift": result["sigma_q_level"],
            "loss": result["loss"],
        }
        for name, result in results.items()
    }
).T
accuracy.loc["seasonal naive", ["MAPE %", "bias %"]] = [mape(naive), bias(naive)]
accuracy.round(3)

In [ ]:
context = 14  # Fit days drawn before the holdout.
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(dates[N_FIT - context :], sales[N_FIT - context :], color=ACTUAL_COLOR, lw=1.2, label="actual")
for name, result in results.items():
    ax.plot(dates[N_FIT:], result["forecast"], color=FIT_COLORS[name], label=name)
ax.axvline(dates[N_FIT], color=CONTEXT_COLOR, lw=0.8)
ax.xaxis.set_major_locator(mdates.DayLocator(bymonthday=(1, 8, 15, 22)))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %-d"))
ax.set_title("TOTAL: the 28-day holdout forecast of each fit")
ax.set_ylabel("units")
ax.legend(frameon=False, ncol=4)
fig.tight_layout()
plt.show()

## 4. Decomposition into four components

Each fit's forecast mean, on the standardized target scale, is the sum of four parts:

- **level:** the filtered level state $\ell_t$.
- **day of week:** the six weekday states on their columns, $w_t^\top d_t$.
- **annual Fourier:** the sine and cosine columns times their two coefficients.
- **events:** the event columns times their coefficients, summed over all events into one line.

Over the fit days the level and weekday states are the filtered path. The covariate coefficients are fixed, so the
annual and event parts use their values on the last fit day throughout. Over the holdout the states are held at their
last fit day values, which is what the forecast does, so the four parts add up to the forecast mean there. The
`no covariates` fit has only the first two parts.

In [ ]:
def components(name):
    # The four components of one fit over every day, on the target scale. The covariate parts are absent without them.
    states = results[name]["states"]
    full = np.vstack([states, np.tile(states[-1], (HORIZON, 1))])  # Hold the last fit day's states over the holdout.
    out = {"level": full[:, 0], "day of week": np.einsum("ti,ti->t", base[:, 1:], full[:, 1:n_base])}
    if FITS[name] is not None:
        beta = states[-1, n_base:]
        for group, label in (("annual", "annual Fourier"), ("event", "events")):
            in_group = COV_GROUPS == group
            out[label] = covariates[:, in_group] @ beta[in_group]
    return pd.DataFrame(out, index=dates)


parts = {name: components(name) for name in FITS}

# The four parts add up to the forecast mean of the design, Z @ states, on the holdout days.
for name, tau in FITS.items():
    last = results[name]["states"][-1]
    Z = design(None if tau is None else 1.0)[N_FIT:]  # Unscaled columns, with the unscaled coefficients.
    assert np.allclose(parts[name].iloc[N_FIT:].sum(axis=1), Z @ last, atol=1e-4), name

In [ ]:
labels = ["level", "day of week", "annual Fourier", "events"]
fig, axes = plt.subplots(len(labels), 1, figsize=(12, 9), sharex=True)
for ax, label in zip(axes, labels):
    for name, frame in parts.items():
        if label in frame:
            ax.plot(dates, frame[label], color=FIT_COLORS[name], lw=0.9, label=name)
    ax.axvspan(dates[N_FIT], dates[-1], color="#f0efe9", zorder=0)
    ax.set_title(label)
    ax.set_ylabel("target scale")
axes[0].legend(frameon=False, ncol=3)
axes[0].set_title("level. With covariates it is the level of a Monday with no event, not of an average day.")
fig.tight_layout()
plt.show()

How much each component moves, as the standard deviation of its values over the fit days. A tight prior shrinks the
annual and event parts toward zero and leaves the level and the weekday states to carry the variation.

In [ ]:
spread = pd.DataFrame({name: frame.iloc[:N_FIT].std() for name, frame in parts.items()}).reindex(labels)
spread.round(3)